# IMPORT REQUIRED LIBRARIES

In [0]:
from pyspark.sql.functions import trim, col, round, unix_timestamp, when
from pyspark.sql.types import TimestampType, IntegerType, DecimalType, StringType

# READ FROM BRONZE DELTA TABLE

In [0]:
df = spark.table("fm_prj1.bronze.work_orders")

# TRANSFORMATION

### TRIM - STRING COLUMNS

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            trim(
                col(field.name)
            )
        )

## DROP DUPLICATED AND EMPTY ROWS

In [0]:
df.dropDuplicates()
df.dropna(how="all")

## TYPE CASTING

In [0]:
# Timestamp columns change dtype from string to TimestampType
timestamp_cols = ["open_date", "response_date", "close_date"]

for t_col in timestamp_cols:
    df = df.withColumn(
        t_col,
        col(t_col).cast(TimestampType())
    )

In [0]:
df = (
    df.withColumn(
        "sla_hours",
        col("sla_hours").cast(IntegerType())
    ).withColumn(
        "cost",
        col("cost").cast(DecimalType(10, 2))
    )
)

## DROP EXCESS COLUMNS

In [0]:
cols_to_drop = ["asset_type", "building", "vendor"]
df = df.drop(*cols_to_drop)
display(df.limit(5))

# DERIVED BUSINESS LOGIC COLUMNS

In [0]:
df = (
    df.withColumn(
        "response_hours",
        round(
            (unix_timestamp("response_date") - unix_timestamp("open_date")) / 3600,
            2
        )
    ).withColumn(
        "close_hours",
        round(
            (unix_timestamp("close_date") - unix_timestamp("open_date")) / 3600,
            2
        )
    ).withColumn(
        "sla_met",
        when(col("close_hours") <= col("sla_hours"), 1).otherwise(0)
    ).withColumn(
        "sla_breach_hours",
        round(
            when(col("sla_met") == 0, col("close_hours") - col("sla_hours")).otherwise(0),
            2
        )
    )
)

In [0]:
display(df.limit(5))

# WRITE TO SILVER DELTA TABLE

In [0]:
print("Writing to silver delta table...")
(
    df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fm_prj1.silver.work_order")
)
print("Succesfully completed writing to silver delta table")